### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="santander_customer_transaction_prediction",
    dataset_year="2019",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/c/santander-customer-transaction-prediction",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c santander-customer-transaction-prediction -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/santander_customer_transaction_prediction && mv train.csv local-data-warehouse/santander_customer_transaction_prediction/
""",
    # References
    academic_reference_bibtex=r"""@misc{Piedra2019SantanderCustomerTransactionPrediction,
  author = {Mercedes Piedra and Sohier Dane and Soraya Jimenez},
  title  = {Santander Customer Transaction Prediction},
  year   = {2019},
  howpublished = {\url{https://kaggle.com/competitions/santander-customer-transaction-prediction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Piedra2019SantanderCustomerTransactionPrediction",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- The data seems to be generated or created for the competition. Most features are perfectly normally distributed.
- Kaggle experts found various uniqueness-based features and had to filter fake samples from the test data. We apply the uniqueness-based features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="target",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# Follow #1 solution https://www.kaggle.com/code/fl2ooo/create-data
# - We do not create the feature that is used for / requires test-time adaption for the fake test samples (that do not exist in the train data).
orig = [f'var_{i}' for i in range(200)]
has_one = [f'var_{i}_has_one' for i in range(200)]
has_zero = [f'var_{i}_has_zero' for i in range(200)]
target = task_mold.target_column_name

for f in orig:
    df[f + '_has_one'] = 0
    df[f + '_has_zero'] = 0
    f_1 = df.loc[df[target] == 1, f].value_counts()

    f_1_1 = set(f_1.index[f_1 > 1])
    f_0_1 = set(f_1.index[f_1 > 0])

    f_0 = df.loc[df[target] == 0, f].value_counts()
    f_0_0 = set(f_0.index[f_0 > 1])
    f_1_0 = set(f_0.index[f_0 > 0])

    df.loc[df[target] == 1, f + '_has_one'] = df.loc[df[target] == 1, f].isin(f_1_1).astype(int)
    df.loc[df[target] == 0, f + '_has_one'] = df.loc[df[target] == 0, f].isin(f_0_1).astype(int)
    df.loc[df[target] == 1, f + '_has_zero'] = df.loc[df[target] == 1, f].isin(f_1_0).astype(int)
    df.loc[df[target] == 0, f + '_has_zero'] = df.loc[df[target] == 0, f].isin(f_0_0).astype(int)
    df = df.copy()

df.loc[:, has_one] = 2*df.loc[:, has_one].values + df.loc[:, has_zero].values

df = df.drop(columns=["ID_code"])

cat_cols = [task_mold.target_column_name] + has_one + has_zero
for col in cat_cols:
    df[col] = df[col].astype("category")

df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()